# Team 04 — Sun Analysis Fitness (Phase 1)

The **avoid-the-worst-sun** capability, built to the same shape as `test_view_analysis.ipynb`
so the two fitnesses compose into **one optimal solution**.

Team method: the sun is **one dominant diagonal vector** (a single azimuth + altitude — the low
west-south-west summer sun as the worst case), not an annual simulation. The agent then places /
orients buildings to *avoid* that sun while keeping their view.

What changed from the first draft: the logic is no longer a flat 2D projection. It now

1. scores **two or more buildings together**, each shading the other,
2. **combines with view analysis** in the NSGA-II optimizer (`unblocked_view` + `sun_avoidance`)
   to produce the Pareto-optimal layout, and
3. presents the result in **3D** — facade cells coloured by sun exposure, with real per-floor
   mutual shading (a tall building shades only the lower floors of its neighbour).

Sections:
1. Scene — site, two buildings with heights, the worst-case sun arrow
2. 2D facade exposure + worst-sun side of the site
3. Combine with view — single-building **view-vs-sun** Pareto front
4. **Two buildings together** — joint view+sun NSGA-II with mutual shading
5. **3D height-aware** sun exposure — per-floor mutual shading (controlled study)
6. **3D plotly** — facade sun-exposure heatmap of the mutual-shading layout

Deterministic except the NSGA-II sections (need `pymoo`) and the 3D scene (needs `plotly`).

In [ ]:
from __future__ import annotations
import sys, math
from pathlib import Path

workspace_root = Path.cwd().resolve()
candidate_roots = (
    workspace_root, workspace_root.parent,
    workspace_root / 'team_04', workspace_root.parent / 'team_04',
)
TEAM_ROOT = next((p for p in candidate_roots if (p / 'agent').exists()), None)
if TEAM_ROOT is None:
    raise FileNotFoundError('Run from workspace root, team_04, or team_04/test_notebooks.')
if str(TEAM_ROOT) not in sys.path:
    sys.path.insert(0, str(TEAM_ROOT))

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

from agent.tools.sun_analysis import (
    compute_sun_vectors, evaluate_sun_exposure, evaluate_sun_exposure_3d,
    identify_worst_sun_side, worst_case_sun_vector, visualize_sun_3d,
    _sun_horizontal_unit, WORST_CASE_PRESETS,
)
from agent.tools.site_model import build_site_model
from agent.tools.building_shape_graph import build_shape_model
print('Team root:', TEAM_ROOT)
print('Sun presets:', list(WORST_CASE_PRESETS))

## 1. Scene — two buildings with heights + the worst-case sun

A 100 × 100 m site, an **L-shaped tower (24 m, 8 floors)** and a **T-shaped block (12 m, 4 floors)**.
The default sun is `summer_west` — a low WSW afternoon sun. Heights matter: the tower will shade
the block's lower floors but not the floors that rise above it.

In [ ]:
SITE = [[0, 0, 0], [100, 0, 0], [100, 100, 0], [0, 100, 0], [0, 0, 0]]

BTYPE_1, AREA_1, HEIGHT_1 = 'L', 675.0, 24.0   # 8-storey tower
BTYPE_2, AREA_2, HEIGHT_2 = 'T', 600.0, 12.0   # 4-storey block

def base_boundary(btype, area):
    poly = build_shape_model(area=area, building_type=btype,
                             building_depth=15.0, shape_ratio=0.5).polygon
    return [[round(float(x), 3), round(float(y), 3), 0.0] for x, y in poly.exterior.coords]

bld_origin_1 = base_boundary(BTYPE_1, AREA_1)
bld_origin_2 = base_boundary(BTYPE_2, AREA_2)

site_model = build_site_model(SITE, {})
sun_vectors = compute_sun_vectors()        # worst-case 'summer_west'
worst = worst_case_sun_vector()
PIECE_LENGTH = 3.0
print('Worst-case sun:', worst)
print(f'B1 {BTYPE_1} {HEIGHT_1:.0f}m   B2 {BTYPE_2} {HEIGHT_2:.0f}m')

In [ ]:
def draw_site(ax, site=SITE):
    ax.add_patch(plt.Polygon([(p[0], p[1]) for p in site[:-1]], fc='#f0ede6', ec='#555', lw=2, zorder=1))
    ax.set_aspect('equal')

def draw_sun_arrow(ax, sun, site=SITE, label=True):
    xs = [p[0] for p in site]; ys = [p[1] for p in site]
    cx, cy = sum(xs)/len(xs), sum(ys)/len(ys)
    span = max(max(xs)-min(xs), max(ys)-min(ys))
    ux, uy = _sun_horizontal_unit(sun['azimuth'])
    start = (cx + ux * span * 0.55, cy + uy * span * 0.55)
    ax.annotate('', xy=(cx, cy), xytext=start, arrowprops=dict(arrowstyle='-|>', color='#f39c12', lw=3), zorder=6)
    if label:
        ax.plot(*start, 'o', color='#f1c40f', ms=18, zorder=6)
        ax.text(start[0], start[1], 'sun', ha='center', va='center', fontsize=8, weight='bold', zorder=7)
        ax.set_title(f"Worst-case sun  az={sun['azimuth']}\u00b0  alt={sun['altitude']}\u00b0")

fig, ax = plt.subplots(figsize=(7, 6))
draw_site(ax)
for bnd, c in ((bld_origin_1, '#16a085'), (bld_origin_2, '#e67e22')):
    ax.add_patch(plt.Polygon([(p[0], p[1]) for p in bnd[:-1]], fc=c, ec='k', lw=1, alpha=0.6))
draw_sun_arrow(ax, worst)
ax.set_xlim(-40, 110); ax.set_ylim(-10, 110)
plt.show()

## 2. 2D facade exposure + worst-sun side

Ground-level facade scoring (0–1, **lower = better**) and the worst site edge — the direction a
sensitive facade or courtyard opening should turn away from.

In [ ]:
result = evaluate_sun_exposure(bld_origin_1, sun_vectors, piece_length=2.0)
side_info = identify_worst_sun_side(site_model, sun_vectors)
print('B1 ground sun_exposure_score:', result['sun_exposure_score'])
print('Worst site side:', side_info['worst_side']['label'], '->', side_info['worst_compass_sector'])

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))
ax = axes[0]
ax.add_patch(plt.Polygon([(p[0], p[1]) for p in bld_origin_1[:-1]], fc='none', ec='#34495e'))
xs = [p['point'][0] for p in result['per_test_point']]
ys = [p['point'][1] for p in result['per_test_point']]
cs = [p['normalized_exposure'] for p in result['per_test_point']]
sc = ax.scatter(xs, ys, c=cs, cmap='YlOrRd', vmin=0, vmax=1, s=70, edgecolors='k', linewidths=0.4)
plt.colorbar(sc, ax=ax, label='sun exposure (0 shaded \u2192 1 hit)')
draw_sun_arrow(ax, worst, site=bld_origin_1, label=False); ax.set_aspect('equal')
ax.set_title('B1 facade test points by exposure')

ax = axes[1]
coords = [(p[0], p[1]) for p in SITE[:-1]]
for s in side_info['per_side']:
    i = s['edge_index']; a = coords[i]; b = coords[(i + 1) % len(coords)]
    sco = s['sun_exposure_score']
    ax.plot([a[0], b[0]], [a[1], b[1]], '-', lw=2 + 8 * sco, color=plt.cm.YlOrRd(0.2 + 0.8 * sco))
    ax.text(*s['midpoint'], f"{s['compass_sector']}\n{sco:.2f}", ha='center', va='center', fontsize=8)
draw_sun_arrow(ax, worst, label=False); ax.set_aspect('equal')
ax.set_title('Site sides by worst-sun exposure (thick/red = worst)')
plt.show()

## 3. Combine with view — single-building view-vs-sun Pareto

The optimizer gains a `sun_avoidance` objective (`1 - sun_exposure`). With `sun_vectors` + a
`sun_weight` and no attractor, the single-building NSGA-II runs a **true two-objective** front:
maximise outward view, minimise worst-sun exposure. Every point is a non-dominated trade-off.

In [ ]:
try:
    from agent.tools.view_optimizer import optimize_view_placement, optimize_two_building_placement
    PYMOO = True
except Exception as exc:
    PYMOO = False
    print('pymoo/view_optimizer unavailable:', exc)

if PYMOO:
    # A western obstacle (also a view blocker) so view and sun genuinely trade off.
    OBSTACLES = [[[104, 20, 0], [120, 20, 0], [120, 80, 0], [104, 80, 0], [104, 20, 0]]]
    single = optimize_view_placement(
        boundary=bld_origin_1, site_boundary=SITE, obstacles=OBSTACLES,
        sun_vectors=sun_vectors, sun_weight=0.6,
        population_size=40, generation_count=50, rotation_step_degrees=15,
        random_seed=42, saved_option_count=14,
    )
    sols = single['pareto_solutions']
    print('objectives:', [(c['name'], round(c['weight'], 2)) for c in single['objective_configs']])
    print(f"{'rank':>4} {'view':>6} {'sun_exp':>8} {'combined':>9} {'rot':>5}")
    for s in sols[:8]:
        print(f"{s['rank']:>4} {s['unblocked_view_score']:>6.3f} {s['sun_exposure_score']:>8.3f} "
              f"{s['combined_score']:>9.3f} {s['rotation_degrees']:>5}")

In [ ]:
if PYMOO and sols:
    fig, ax = plt.subplots(figsize=(6.5, 5))
    v = [s['unblocked_view_score'] for s in sols]
    e = [s['sun_exposure_score'] for s in sols]
    sc = ax.scatter(v, e, c=[s['combined_score'] for s in sols], cmap='viridis', s=80, edgecolors='k')
    plt.colorbar(sc, ax=ax, label='combined (view + sun-avoidance)')
    for s in sols:
        ax.annotate(f"{s['rotation_degrees']}\u00b0", (s['unblocked_view_score'], s['sun_exposure_score']),
                    textcoords='offset points', xytext=(4, 3), fontsize=7)
    ax.set_xlabel('view score (higher better) \u2192')
    ax.set_ylabel('\u2190 sun exposure (lower better)')
    ax.invert_yaxis()
    ax.set_title('Single-building Pareto: outward view vs. worst-sun exposure')
    plt.show()

## 4. Two buildings together — joint view + sun, with mutual shading

The practical case. `optimize_two_building_placement` takes the **same** `sun_vectors` + `sun_weight`;
each building is already passed as the other's obstacle, so it shades the other for both view **and**
sun. The combined score per building weights view against sun-avoidance; NSGA-II finds the layout
where *both* buildings keep their view and dodge the worst sun.

In [ ]:
if PYMOO:
    print('Running two-building joint view+sun NSGA-II (1-3 min)...')
    joint = optimize_two_building_placement(
        boundary_1=bld_origin_1, boundary_2=bld_origin_2,
        site_boundary=SITE, external_obstacles=OBSTACLES,
        sun_vectors=sun_vectors, sun_weight=0.6,
        min_building_separation=6.0,
        population_size=50, generation_count=70, rotation_step_degrees=15,
        random_seed=42, saved_option_count=12,
    )
    jsols = joint['pareto_solutions']
    print('objectives:', [(c['name'], round(c['weight'], 2)) for c in joint['objective_configs']])
    print(f'Pareto solutions: {len(jsols)}   best avg combined: {joint["best_avg_combined_score"]:.3f}')
    print(f"{'rank':>4}  {'avg':>6} | {'B1view':>6} {'B1sun':>6} | {'B2view':>6} {'B2sun':>6} | {'clear':>5}")
    for s in jsols[:8]:
        b1, b2 = s['building_1'], s['building_2']
        print(f"{s['rank']:>4}  {s['avg_combined_score']:>6.3f} | "
              f"{b1['unblocked_view_score']:>6.2f} {b1['sun_exposure_score']:>6.2f} | "
              f"{b2['unblocked_view_score']:>6.2f} {b2['sun_exposure_score']:>6.2f} | "
              f"{s['clearance_between_buildings_m']:>5.1f}")

In [ ]:
if PYMOO and jsols:
    def plot_layout(ax, sol, title):
        draw_site(ax)
        for obs in OBSTACLES:
            ax.add_patch(plt.Polygon([(p[0], p[1]) for p in obs[:-1]], fc='#c0392b', ec='#333', alpha=0.7))
        for b, c in ((sol['building_1'], '#2980b9'), (sol['building_2'], '#e67e22')):
            det = evaluate_sun_exposure(b['boundary'], sun_vectors, piece_length=PIECE_LENGTH)
            ax.add_patch(plt.Polygon([(p[0], p[1]) for p in b['boundary'][:-1]], fc=c, ec='k', lw=1.2, alpha=0.4))
            px = [p['point'][0] for p in det['per_test_point']]
            py = [p['point'][1] for p in det['per_test_point']]
            pc = [p['normalized_exposure'] for p in det['per_test_point']]
            ax.scatter(px, py, c=pc, cmap='YlOrRd', vmin=0, vmax=1, s=22, edgecolors='k', linewidths=0.2, zorder=4)
        draw_sun_arrow(ax, worst, label=False)
        ax.set_xlim(-40, 120); ax.set_ylim(-10, 110); ax.set_title(title, fontsize=9)

    fig, axes = plt.subplots(1, 3, figsize=(20, 6.5))
    for ax, sol in zip(axes, jsols[:3]):
        b1, b2 = sol['building_1'], sol['building_2']
        plot_layout(ax, sol,
                    f"rank #{sol['rank']}  avg={sol['avg_combined_score']:.3f}\n"
                    f"B1 view {b1['unblocked_view_score']:.2f} sun {b1['sun_exposure_score']:.2f}   "
                    f"B2 view {b2['unblocked_view_score']:.2f} sun {b2['sun_exposure_score']:.2f}")
    handles = [mpatches.Patch(color='#2980b9', label='Building 1 (tower)'),
               mpatches.Patch(color='#e67e22', label='Building 2 (block)'),
               mpatches.Patch(color='#c0392b', label='External obstacle')]
    fig.legend(handles=handles, loc='lower center', ncol=3, fontsize=9)
    fig.suptitle('Top joint layouts \u2014 facade points coloured by worst-sun exposure', y=1.02)
    plt.show()

## 5. 3D height-aware sun exposure — real per-floor mutual shading

The flat 2D score treats a neighbour as a ground-level wall. In 3D each facade is a grid of cells
(one per floor); an obstacle of height `h` only shades a cell at height `z` when `h > z`.

To make the physics **legible** we use a controlled placement here: the **shorter 12 m block sits
on the tower's sun (WSW) side**. The tower's lower floors (below 12 m) fall in the block's shadow,
while the floors that rise above 12 m clear it and keep full sun — a graded shadow no flat 2D
projection can produce. (Section 4's *optimizer* does the opposite: it moves buildings apart to
avoid exactly this — which is the point. Here we force the overlap to show the mechanism.)

In [ ]:
# Controlled mutual-shading layout: put the SHORTER block on the TALLER tower's
# sun side, so the tower's lower floors fall in shadow and its upper floors clear it.
from shapely import affinity
from shapely.geometry import Polygon as _P

def _centroid(bnd):
    p = _P([(q[0], q[1]) for q in bnd[:-1]])
    return p.centroid.x, p.centroid.y

def place_at(bnd, tx, ty):
    cx, cy = _centroid(bnd)
    poly = affinity.translate(_P([(q[0], q[1]) for q in bnd[:-1]]), tx - cx, ty - cy)
    return [[round(x, 3), round(y, 3), 0.0] for x, y in poly.exterior.coords]

ux, uy = _sun_horizontal_unit(worst['azimuth'])   # horizontal direction TOWARD the sun (WSW)
TOWER_AT = (66, 52)
BLOCK_AT = (TOWER_AT[0] + ux * 28, TOWER_AT[1] + uy * 28)   # block between tower and sun
bnd1 = place_at(bld_origin_1, *TOWER_AT)   # 24 m tower — the SHADED building
bnd2 = place_at(bld_origin_2, *BLOCK_AT)   # 12 m block — the SHADING neighbour

# Tower shaded by the block (real heights); block shaded by the tower.
sun3d_1 = evaluate_sun_exposure_3d(bnd1, HEIGHT_1, sun_vectors,
    [{'boundary': bnd2, 'height': HEIGHT_2}], piece_length=PIECE_LENGTH, floor_height=3.0)
sun3d_2 = evaluate_sun_exposure_3d(bnd2, HEIGHT_2, sun_vectors,
    [{'boundary': bnd1, 'height': HEIGHT_1}], piece_length=PIECE_LENGTH, floor_height=3.0)

for name, r in (('B1 tower (24 m, shaded by block)', sun3d_1), ('B2 block (12 m)', sun3d_2)):
    print(f"\n{name}: whole-building sun_exposure_score_3d = {r['sun_exposure_score_3d']:.3f} (lower better)")
    for f in r['per_floor']:
        bar = '#' * int(f['sun_exposure_score'] * 40)
        flag = '  <- below 12 m: in shadow' if f['z_level'] < HEIGHT_2 else ''
        print(f"  floor {f['floor_number']:>2} z={f['z_level']:>4.1f}m  exp={f['sun_exposure_score']:.3f}  |{bar:<40}|{flag}")

In [ ]:
# Per-floor exposure profile — see the shadow lift off the upper floors.
fig, ax = plt.subplots(figsize=(7, 5))
for r, name, c in ((sun3d_1, 'B1 tower (24 m)', '#2980b9'), (sun3d_2, 'B2 block (12 m)', '#e67e22')):
    z = [f['z_level'] for f in r['per_floor']]
    e = [f['sun_exposure_score'] for f in r['per_floor']]
    ax.plot(e, z, 'o-', color=c, label=name)
ax.set_xlabel('floor sun exposure (lower better) \u2192')
ax.set_ylabel('height z (m)')
ax.set_title('Sun exposure vs. height \u2014 lower floors sit in the neighbour\u2019s shadow')
ax.legend(); ax.grid(alpha=0.2)
plt.show()

## 6. 3D plotly — facade sun-exposure heatmap

The controlled two-building layout from section 5 in 3D. Every side face is a continuous heatmap
(blue = shaded / cool → red = full worst-sun hit) computed with real mutual shading, and the
dominant sun vector is drawn at its true altitude. Rotate it: the block's shadow reads as a cool
band across the **tower's lower floors**, while the tower's upper floors glow red above the block.

In [ ]:
try:
    fig3d = visualize_sun_3d(
        site_boundary=SITE,
        buildings=[
            {'boundary': bnd1, 'height': HEIGHT_1, 'label': f'B1 tower ({HEIGHT_1:.0f} m)'},
            {'boundary': bnd2, 'height': HEIGHT_2, 'label': f'B2 block ({HEIGHT_2:.0f} m)'},
        ],
        sun_vectors=sun_vectors,
        sun_results=[sun3d_1, sun3d_2],
        buildable_zone_boundary=(site_model.get('setbacks') or {}).get('buildable_boundary'),
        piece_length=PIECE_LENGTH, floor_height=3.0,
        title='3D Sun Exposure \u2014 facades coloured by worst-sun hit (blue=shaded \u2192 red=hit)',
    )
    fig3d.show()
except RuntimeError as exc:
    print('plotly not available:', exc)

## 7. Complex site — grid-aligned placement reacting to the sun

Sections 1–6 used a 100 × 100 square. Here we bring in the **grid-alignment logic** on a splayed,
non-orthogonal site and combine it with the sun fitness:

1. Pick the side **chosen by sun analysis** (the worst-sun frontage) and draw a **straight grid**
   parallel + perpendicular to it.
2. Place a **U** building **rigidly, by function** (`place_building_by_function`) — residential, so its
   long side runs **perpendicular** to the chosen side; it stays a recognisable, grid-aligned shape.
3. Score each in-site placement by **worst-sun exposure** (`evaluate_sun_exposure`) and keep the one
   that both fits the splayed site **and** dodges the worst sun.

The two capabilities composed: the building reacts to the *site* (straight grid keyed to the
sun-chosen side) **and** the *sun* (exposure fitness) at once, while staying a realistic building.

In [ ]:
from agent.tools.site_grid import derive_site_grid, place_building_by_function, frontage_for_function
from agent.tools.view_analysis import _coerce_polygon_2d

# A splayed, non-orthogonal site. Pick the side CHOSEN BY SUN ANALYSIS (here the worst
# sun side — the frontage to set the grid against) and draw a STRAIGHT grid parallel +
# perpendicular to it. Buildings are placed rigidly, oriented by function.
CSITE = [[0, 0, 0], [130, 18, 0], [150, 92, 0], [62, 128, 0], [-14, 74, 0], [0, 0, 0]]
csite_model = build_site_model(CSITE, {'default_setback': 6.0})
cworst = identify_worst_sun_side(csite_model, sun_vectors)
chosen_idx = max(cworst['per_side'], key=lambda s: s['sun_exposure_score'])['edge_index']
cgrid = derive_site_grid(csite_model, spacing=12.0, alignment_side=chosen_idx)   # STRAIGHT grid
csite_poly = _coerce_polygon_2d(CSITE)
print(f"chosen side based on sun analysis: side {chosen_idx} ({cworst['worst_compass_sector']}); grid angle {cgrid['angle_deg']:.1f} deg")

def draw_csite(ax):
    ax.plot([p[0] for p in CSITE], [p[1] for p in CSITE], color='#e23b2e', lw=2.2, zorder=5)
    ax.set_aspect('equal')

def draw_cgrid(ax, color='#7fbf3f'):
    for ln in cgrid['grid_lines']:                       # straight segments
        ax.plot([ln[0][0], ln[1][0]], [ln[0][1], ln[1][1]], color=color, lw=0.7, zorder=2)

cs = [(p[0], p[1]) for p in CSITE[:-1]]
fig, ax = plt.subplots(figsize=(8, 7))
draw_csite(ax); draw_cgrid(ax)
a, b = cs[chosen_idx], cs[(chosen_idx + 1) % len(cs)]
ax.plot([a[0], b[0]], [a[1], b[1]], color='#00bcd4', lw=6, zorder=4)
ax.annotate('chosen side based on sun analysis', xy=((a[0]+b[0])/2, (a[1]+b[1])/2), xytext=(a[0]+4, a[1]-8),
            rotation=math.degrees(math.atan2(b[1]-a[1], b[0]-a[0])), fontsize=9, color='#006064', zorder=8)
draw_sun_arrow(ax, worst, site=CSITE, label=True)
ax.set_title('Complex site + STRAIGHT grid keyed to the sun-chosen side')
ax.set_xlim(-30, 170); ax.set_ylim(-20, 145)
plt.show()

In [ ]:
# With no obstacles, a building's worst-sun exposure depends on its ORIENTATION,
# which the function fixes. So we compare the SAME U placed commercial (long side
# parallel to the sun-chosen side) vs residential (perpendicular): the function/
# orientation changes how much worst-sun the facades catch. The grid keeps both
# rigidly aligned; the agent picks the function (or lets sun pick the orientation).
u_base = base_boundary('U', 600.0)
u_area = _coerce_polygon_2d(u_base).area
ccen = [sum(p[0] for p in CSITE[:-1]) / 5, sum(p[1] for p in CSITE[:-1]) / 5]

def place_func_inside(base, func):
    for nd in sorted(cgrid['grid_nodes'], key=lambda n: (n[0]-ccen[0])**2 + (n[1]-ccen[1])**2):
        placed = place_building_by_function(base, cgrid, nd, func)
        if csite_poly.contains(_coerce_polygon_2d(placed)):
            return placed
    return None

fig, axes = plt.subplots(1, 2, figsize=(17, 7))
scores = {}
for ax, func, col in ((axes[0], 'commercial', '#c0392b'), (axes[1], 'residential', '#2980b9')):
    placed = place_func_inside(u_base, func)
    det = evaluate_sun_exposure(placed, sun_vectors, piece_length=3.0)
    scores[func] = det['sun_exposure_score']
    draw_csite(ax); draw_cgrid(ax)
    a, b = cs[chosen_idx], cs[(chosen_idx + 1) % len(cs)]
    ax.plot([a[0], b[0]], [a[1], b[1]], color='#00bcd4', lw=6, zorder=4)
    ax.add_patch(plt.Polygon([(p[0], p[1]) for p in placed[:-1]], fc=col, ec='k', lw=1.3, alpha=0.4, zorder=6))
    px = [p['point'][0] for p in det['per_test_point']]; py = [p['point'][1] for p in det['per_test_point']]
    pc = [p['normalized_exposure'] for p in det['per_test_point']]
    ax.scatter(px, py, c=pc, cmap='YlOrRd', vmin=0, vmax=1, s=20, edgecolors='k', linewidths=0.2, zorder=7)
    draw_sun_arrow(ax, worst, site=CSITE, label=False)
    ax.set_title(f"{func} ({frontage_for_function(func)} to side)\nworst-sun exposure = {scores[func]:.3f}")
    ax.set_xlim(-30, 170); ax.set_ylim(-20, 145)
better = min(scores, key=scores.get)
print(f"commercial (parallel)     worst-sun exposure: {scores['commercial']:.3f}")
print(f"residential (perpendicular) worst-sun exposure: {scores['residential']:.3f}")
print(f"=> '{better}' orientation catches less worst-sun (the function/orientation drives sun exposure)")
fig.suptitle('Function orientation drives worst-sun exposure (straight grid keyed to the sun-chosen side)', y=1.02)
plt.show()

## Summary

- The sun is **one diagonal vector** (`compute_sun_vectors` / `WORST_CASE_PRESETS`).
- **Two buildings are scored together**: the optimizer's `sun_avoidance` objective combines with
  `unblocked_view`, and each building shades the other (mutual shading) — so the Pareto layout is
  optimal for view *and* sun at once, not one at a time.
- **3D is exact**: `evaluate_sun_exposure_3d` does per-floor shading (a tall building shades only
  the lower floors of its neighbour); `visualize_sun_3d` renders the facade heatmap.
- **Complex site (§7)**: the sun fitness composes with the **grid-alignment** logic — a **straight**
  grid is keyed to the side chosen by sun analysis, a building is placed **rigidly by function**
  (`place_building_by_function`, shape kept), and each in-site placement is scored by worst-sun
  exposure, so the building reacts to the *site* and the *sun* at once while staying realistic.
- **Two buildings — U + H (§8)**: U (commercial, 21 m / 7 floors) is placed **parallel** to the
  sun-chosen side; H (residential, 15 m / 5 floors) is placed **perpendicular** to it. Both are
  fully grid-aligned with real 3D mutual shading — the taller U casts a per-floor shadow on H's
  lower floors, visible in the Plotly facade heatmap (blue = shaded → red = full sun hit).

Backend: `agent/tools/sun_analysis.py` (`evaluate_sun_exposure`, `evaluate_sun_exposure_3d`,
`identify_worst_sun_side`, `visualize_sun_3d`) + the `sun_avoidance` objective in
`view_optimizer.py`, composed with `agent/tools/site_grid.py` grid-aligned function placement.
Regressions: `benchmarking/test_sun_analysis.py` (23 tests, incl. the grid + sun integration).
Frontend overlay: `frontend/site/SunOverlay.tsx` via `POST /tools/{sun_vectors,sun_exposure,sun_exposure_3d,worst_sun_side}`.

In [ ]:
# § 8a — Place U (commercial, 21 m) + H (residential, 15 m) on the complex site.
#         Grid already computed in §7a (cgrid keyed to sun-chosen side).
U_HEIGHT = 21.0   # 7 floors × 3 m
H_HEIGHT = 15.0   # 5 floors × 3 m
PIECE_L8 = 3.0
FLOOR_H8 = 3.0

u8_base = base_boundary('U', 600.0)
h8_base = base_boundary('H', 550.0)

# Site centroid
cx8 = sum(p[0] for p in CSITE[:-1]) / (len(CSITE) - 1)
cy8 = sum(p[1] for p in CSITE[:-1]) / (len(CSITE) - 1)

def _place_func8(base, func, avoid_poly=None, target_dist=None):
    """Find the nearest in-site grid node for `func` orientation; optionally avoid a polygon."""
    if target_dist is None:
        key_fn = lambda n: (n[0] - cx8)**2 + (n[1] - cy8)**2
    else:
        acx, acy = avoid_poly.centroid.x, avoid_poly.centroid.y
        key_fn = lambda n: abs((n[0] - acx)**2 + (n[1] - acy)**2 - target_dist**2)
    for nd in sorted(cgrid['grid_nodes'], key=key_fn):
        placed = place_building_by_function(base, cgrid, nd, func)
        poly = _coerce_polygon_2d(placed)
        if not csite_poly.contains(poly):
            continue
        if avoid_poly is not None and avoid_poly.buffer(4.0).intersects(poly):
            continue
        return placed
    return None

# U — commercial (parallel to chosen side) → place near site centroid
bnd_u = _place_func8(u8_base, 'commercial')
u8_poly = _coerce_polygon_2d(bnd_u)

# H — residential (perpendicular) → target ~40 m from U, no overlap
bnd_h = _place_func8(h8_base, 'residential', avoid_poly=u8_poly, target_dist=40.0)
h8_poly = _coerce_polygon_2d(bnd_h)

# ── 2D overview ──────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 8))
draw_csite(ax)
draw_cgrid(ax)

a8, b8 = cs[chosen_idx], cs[(chosen_idx + 1) % len(cs)]
ax.plot([a8[0], b8[0]], [a8[1], b8[1]], color='#00bcd4', lw=6, zorder=4)

ax.add_patch(plt.Polygon([(p[0], p[1]) for p in bnd_u[:-1]],
             fc='#c0392b', ec='k', lw=1.5, alpha=0.55, zorder=6))
ax.add_patch(plt.Polygon([(p[0], p[1]) for p in bnd_h[:-1]],
             fc='#2980b9', ec='k', lw=1.5, alpha=0.55, zorder=6))

for bnd, lbl in ((bnd_u, 'U'), (bnd_h, 'H')):
    pts = bnd[:-1]
    ax.text(sum(p[0] for p in pts) / len(pts), sum(p[1] for p in pts) / len(pts),
            lbl, ha='center', va='center', fontsize=14, weight='bold', color='white', zorder=8)

draw_sun_arrow(ax, worst, site=CSITE, label=True)

handles = [
    mpatches.Patch(color='#c0392b',
                   label=f'U — commercial, {frontage_for_function("commercial")} to chosen side ({U_HEIGHT:.0f} m)'),
    mpatches.Patch(color='#2980b9',
                   label=f'H — residential, {frontage_for_function("residential")} to chosen side ({H_HEIGHT:.0f} m)'),
    plt.Line2D([0], [0], color='#00bcd4', lw=4, label='Sun-chosen side'),
]
ax.legend(handles=handles, loc='lower right', fontsize=9)
ax.set_title(f'§8 — U + H on complex site; straight grid keyed to side {chosen_idx} ({cworst["worst_compass_sector"]})')
ax.set_xlim(-30, 170); ax.set_ylim(-20, 145)
plt.show()

# 2D ground-level sun scores
s2d_u = evaluate_sun_exposure(bnd_u, sun_vectors, piece_length=PIECE_L8)
s2d_h = evaluate_sun_exposure(bnd_h, sun_vectors, piece_length=PIECE_L8)
print(f"U commercial  (parallel)       2D worst-sun score: {s2d_u['sun_exposure_score']:.3f}  (lower = better)")
print(f"H residential (perpendicular)  2D worst-sun score: {s2d_h['sun_exposure_score']:.3f}  (lower = better)")

In [ ]:
# § 8b — 3D mutual sun exposure + Plotly facade heatmap.
#         Each building treats the other as an obstacle; per-floor shading is real.
sun3d_u = evaluate_sun_exposure_3d(
    bnd_u, U_HEIGHT, sun_vectors,
    [{'boundary': bnd_h, 'height': H_HEIGHT}],
    piece_length=PIECE_L8, floor_height=FLOOR_H8,
)
sun3d_h = evaluate_sun_exposure_3d(
    bnd_h, H_HEIGHT, sun_vectors,
    [{'boundary': bnd_u, 'height': U_HEIGHT}],
    piece_length=PIECE_L8, floor_height=FLOOR_H8,
)

print(f"U commercial  ({U_HEIGHT:.0f} m, {sun3d_u['n_floors']} floors)  3D score: {sun3d_u['sun_exposure_score_3d']:.3f}  (lower = better)")
for f in sun3d_u['per_floor']:
    bar = '#' * int(f['sun_exposure_score'] * 30)
    print(f"  floor {f['floor_number']:>2}  z={f['z_level']:>5.1f} m  exp={f['sun_exposure_score']:.3f}  |{bar}")

print(f"\nH residential ({H_HEIGHT:.0f} m, {sun3d_h['n_floors']} floors)  3D score: {sun3d_h['sun_exposure_score_3d']:.3f}  (lower = better)")
for f in sun3d_h['per_floor']:
    bar = '#' * int(f['sun_exposure_score'] * 30)
    shadow = '  <- in U shadow' if f['z_level'] < U_HEIGHT and f['sun_exposure_score'] < sun3d_h['sun_exposure_score_3d'] else ''
    print(f"  floor {f['floor_number']:>2}  z={f['z_level']:>5.1f} m  exp={f['sun_exposure_score']:.3f}  |{bar}{shadow}")

# ── Plotly 3D facade heatmap ──────────────────────────────────────────────────
try:
    fig3d = visualize_sun_3d(
        site_boundary=CSITE,
        buildings=[
            {'boundary': bnd_u, 'height': U_HEIGHT, 'label': f'U — commercial ({U_HEIGHT:.0f} m)'},
            {'boundary': bnd_h, 'height': H_HEIGHT, 'label': f'H — residential ({H_HEIGHT:.0f} m)'},
        ],
        sun_vectors=sun_vectors,
        sun_results=[sun3d_u, sun3d_h],
        piece_length=PIECE_L8,
        floor_height=FLOOR_H8,
        title='3D Sun Exposure — U (commercial) + H (residential) on complex site',
    )
    fig3d.show()
except RuntimeError as exc:
    print('plotly not available:', exc)

## Summary

- The sun is **one diagonal vector** (`compute_sun_vectors` / `WORST_CASE_PRESETS`).
- **Two buildings are scored together**: the optimizer's `sun_avoidance` objective combines with
  `unblocked_view`, and each building shades the other (mutual shading) — so the Pareto layout is
  optimal for view *and* sun at once, not one at a time.
- **3D is exact**: `evaluate_sun_exposure_3d` does per-floor shading (a tall building shades only
  the lower floors of its neighbour); `visualize_sun_3d` renders the facade heatmap.
- **Complex site (§7)**: the sun fitness composes with the **grid-alignment** logic — a **straight**
  grid is keyed to the side chosen by sun analysis, a building is placed **rigidly by function**
  (`place_building_by_function`, shape kept), and each in-site placement is scored by worst-sun
  exposure, so the building reacts to the *site* and the *sun* at once while staying realistic.

Backend: `agent/tools/sun_analysis.py` (`evaluate_sun_exposure`, `evaluate_sun_exposure_3d`,
`identify_worst_sun_side`, `visualize_sun_3d`) + the `sun_avoidance` objective in
`view_optimizer.py`, composed with `agent/tools/site_grid.py` grid-aligned function placement.
Regressions: `benchmarking/test_sun_analysis.py` (23 tests, incl. the grid + sun integration).
Frontend overlay: `frontend/site/SunOverlay.tsx` via `POST /tools/{sun_vectors,sun_exposure,sun_exposure_3d,worst_sun_side}`.